# Zgodovinske značilke

Nalozimo osnovne znacilke

In [1]:
import pandas as pd
import numpy as np

from collections import defaultdict, deque
from pathlib import Path

features = pd.read_csv("./data/processed/osnovne_znacilke.csv")

features["tourney_date"] = pd.to_datetime(features["tourney_date"])
features["year"] = features["tourney_date"].dt.year

features = features.sort_values(
    ["tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

features.head()

,tourney_id,tourney_name,tourney_date,year,match_num,p1_id,p1_name,p2_id,p2_name,target,...,tourney_level,best_of,round,hand_matchup,rank_diff,rank_points_diff,age_diff,height_diff,round_num,draw_size
0,2000-339,Adelaide,2000-01-03,2000,1,103096,Arnaud Clement,102358,Thomas Enqvist,0,...,A,3,R32,R_vs_R,52.0,-1801.0,-3.7,-17.0,3,32
1,2000-339,Adelaide,2000-01-03,2000,2,103819,Roger Federer,102533,Jens Knippschild,1,...,A,3,R32,R_vs_R,-27.0,224.0,-6.5,-5.0,3,32
2,2000-339,Adelaide,2000-01-03,2000,3,101885,Wayne Arthurs,102998,Jan Michael Gambill,0,...,A,3,R32,L_vs_R,47.0,-354.0,6.2,0.0,3,32
3,2000-339,Adelaide,2000-01-03,2000,4,102776,Andrew Ilie,103206,Sebastien Grosjean,0,...,A,3,R32,R_vs_R,27.0,-453.0,2.1,5.0,3,32
4,2000-339,Adelaide,2000-01-03,2000,5,102796,Magnus Norman,102401,Scott Draper,1,...,A,3,R32,R_vs_L,-139.0,1451.0,-2.0,10.0,3,32


### Pomožne funkcije

`expected_score` izračuna pričakovan rezultat dvoboja po Elo formuli, `update_elo` posodobi ratinga obeh igralcev glede na izid. `winrate_from_history` in `safe_rate` sta varni različici za primere, ko zgodovina še ni dostopna.

In [2]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(rating_a, rating_b, score_a, k=32):
    """
    score_a = 1, če igralec A zmaga
    score_a = 0, če igralec A izgubi
    """
    exp_a = expected_score(rating_a, rating_b)
    exp_b = 1 - exp_a

    score_b = 1 - score_a

    new_a = rating_a + k * (score_a - exp_a)
    new_b = rating_b + k * (score_b - exp_b)

    return new_a, new_b


def winrate_from_history(history, default=0.5):
    if len(history) == 0:
        return default
    return np.mean(history)


def safe_rate(wins, matches, default=0.5):
    if matches == 0:
        return default
    return wins / matches

def make_pair_key(player_a, player_b):
    return tuple(sorted([str(player_a), str(player_b)]))


def make_surface_pair_key(player_a, player_b, surface):
    return (make_pair_key(player_a, player_b), surface)


def h2h_winrate(wins, matches, default=0.5):
    if matches == 0:
        return default
    return wins / matches

### Inicializacija

In [3]:
INITIAL_ELO = 1500.0
K_FACTOR = 32

### Izračun zgodovinskih značilk

Tekme obdelujemo kronološko. Za vsako tekmo najprej zapišemo trenutne vrednosti (Elo, forma, winrate) PRED tekmo, šele nato posodobimo statistike z izidom. Tekme znotraj istega `tourney_date` obravnavamo skupaj, da ne pride do data leakage znotraj enega turnirskega tedna.

In [4]:
# Elo rating igralca
elo = defaultdict(lambda: INITIAL_ELO)

# Elo rating igralca na posamezni površini
surface_elo = defaultdict(lambda: INITIAL_ELO)

# Zadnjih 10 rezultatov igralca
last10_results = defaultdict(lambda: deque(maxlen=10))

# Zadnjih 10 rezultatov igralca na posamezni površini
surface_last10_results = defaultdict(lambda: deque(maxlen=10))

# Skupni rezultati
total_matches = defaultdict(int)
total_wins = defaultdict(int)

# Rezultati po površinah
surface_matches = defaultdict(int)
surface_wins = defaultdict(int)

# Head-to-head skupno
h2h_matches = defaultdict(int)
h2h_wins = defaultdict(lambda: defaultdict(int))

# Head-to-head po površini
surface_h2h_matches = defaultdict(int)
surface_h2h_wins = defaultdict(lambda: defaultdict(int))

# Zadnji medsebojni rezultat
# Vrednost bo ID igralca, ki je dobil zadnji medsebojni dvoboj
last_h2h_winner = {}

historical_rows = []

# Pomembno:
# Ker tourney_date pri Sackmannu pogosto pomeni začetek tedna turnirja,
# vse tekme z istim datumom najprej dobijo značilke,
# šele potem posodobimo ratinge.
for date, group in features.groupby("tourney_date", sort=True):

    pending_updates = []

    for idx, row in group.iterrows():
        p1 = row["p1_id"]
        p2 = row["p2_id"]
        surface = row["surface"]
        target = int(row["target"])

        p1_surface_key = (p1, surface)
        p2_surface_key = (p2, surface)
        
        pair_key = make_pair_key(p1, p2)
        surface_pair_key = make_surface_pair_key(p1, p2, surface)

        # -------------------------
        # H2H vrednosti PRED tekmo
        # -------------------------

        previous_h2h_matches = h2h_matches[pair_key]
        previous_p1_h2h_wins = h2h_wins[pair_key][str(p1)]
        previous_p2_h2h_wins = h2h_wins[pair_key][str(p2)]

        p1_h2h_winrate = h2h_winrate(
            previous_p1_h2h_wins,
            previous_h2h_matches,
            default=0.5
        )

        p2_h2h_winrate = h2h_winrate(
            previous_p2_h2h_wins,
            previous_h2h_matches,
            default=0.5
        )

        previous_surface_h2h_matches = surface_h2h_matches[surface_pair_key]
        previous_p1_surface_h2h_wins = surface_h2h_wins[surface_pair_key][str(p1)]
        previous_p2_surface_h2h_wins = surface_h2h_wins[surface_pair_key][str(p2)]

        p1_surface_h2h_winrate = h2h_winrate(
            previous_p1_surface_h2h_wins,
            previous_surface_h2h_matches,
            default=0.5
        )

        p2_surface_h2h_winrate = h2h_winrate(
            previous_p2_surface_h2h_wins,
            previous_surface_h2h_matches,
            default=0.5
        )

        if pair_key not in last_h2h_winner:
            last_h2h_result = 0.5
        elif last_h2h_winner[pair_key] == str(p1):
            last_h2h_result = 1.0
        else:
            last_h2h_result = 0.0

        # -------------------------
        # Vrednosti PRED tekmo
        # -------------------------

        p1_elo = elo[p1]
        p2_elo = elo[p2]

        p1_surface_elo = surface_elo[p1_surface_key]
        p2_surface_elo = surface_elo[p2_surface_key]

        p1_last10 = winrate_from_history(last10_results[p1])
        p2_last10 = winrate_from_history(last10_results[p2])

        p1_surface_last10 = winrate_from_history(surface_last10_results[p1_surface_key])
        p2_surface_last10 = winrate_from_history(surface_last10_results[p2_surface_key])

        p1_total_winrate = safe_rate(total_wins[p1], total_matches[p1])
        p2_total_winrate = safe_rate(total_wins[p2], total_matches[p2])

        p1_surface_winrate = safe_rate(
            surface_wins[p1_surface_key],
            surface_matches[p1_surface_key]
        )
        p2_surface_winrate = safe_rate(
            surface_wins[p2_surface_key],
            surface_matches[p2_surface_key]
        )

        p1_matches_played = total_matches[p1]
        p2_matches_played = total_matches[p2]

        p1_surface_matches_played = surface_matches[p1_surface_key]
        p2_surface_matches_played = surface_matches[p2_surface_key]

        historical_rows.append({
            "row_id": idx,

            "p1_elo": p1_elo,
            "p2_elo": p2_elo,
            "elo_diff": p1_elo - p2_elo,

            "p1_surface_elo": p1_surface_elo,
            "p2_surface_elo": p2_surface_elo,
            "surface_elo_diff": p1_surface_elo - p2_surface_elo,

            "p1_last10_winrate": p1_last10,
            "p2_last10_winrate": p2_last10,
            "last10_winrate_diff": p1_last10 - p2_last10,

            "p1_surface_last10_winrate": p1_surface_last10,
            "p2_surface_last10_winrate": p2_surface_last10,
            "surface_last10_winrate_diff": p1_surface_last10 - p2_surface_last10,

            "p1_total_winrate": p1_total_winrate,
            "p2_total_winrate": p2_total_winrate,
            "total_winrate_diff": p1_total_winrate - p2_total_winrate,

            "p1_surface_winrate": p1_surface_winrate,
            "p2_surface_winrate": p2_surface_winrate,
            "surface_winrate_diff": p1_surface_winrate - p2_surface_winrate,

            "p1_matches_played": p1_matches_played,
            "p2_matches_played": p2_matches_played,
            "matches_played_diff": p1_matches_played - p2_matches_played,

            "p1_surface_matches_played": p1_surface_matches_played,
            "p2_surface_matches_played": p2_surface_matches_played,
            "surface_matches_played_diff": (
                p1_surface_matches_played - p2_surface_matches_played
            ),
            
            "h2h_matches": previous_h2h_matches,
            "h2h_p1_wins": previous_p1_h2h_wins,
            "h2h_p2_wins": previous_p2_h2h_wins,
            "h2h_winrate_p1": p1_h2h_winrate,
            "h2h_winrate_diff": p1_h2h_winrate - p2_h2h_winrate,

            "surface_h2h_matches": previous_surface_h2h_matches,
            "surface_h2h_p1_wins": previous_p1_surface_h2h_wins,
            "surface_h2h_p2_wins": previous_p2_surface_h2h_wins,
            "surface_h2h_winrate_p1": p1_surface_h2h_winrate,
            "surface_h2h_winrate_diff": (
                p1_surface_h2h_winrate - p2_surface_h2h_winrate
            ),

            "last_h2h_result": last_h2h_result,
        })

        pending_updates.append((p1, p2, surface, target))

    # -------------------------
    # Posodobitve PO tekmah tega datuma
    # -------------------------

    for p1, p2, surface, target in pending_updates:
        p1_score = target
        p2_score = 1 - target

        p1_surface_key = (p1, surface)
        p2_surface_key = (p2, surface)

        # update splošnega Elo
        new_p1_elo, new_p2_elo = update_elo(
            elo[p1],
            elo[p2],
            p1_score,
            k=K_FACTOR
        )

        elo[p1] = new_p1_elo
        elo[p2] = new_p2_elo

        # update surface Elo
        new_p1_surface_elo, new_p2_surface_elo = update_elo(
            surface_elo[p1_surface_key],
            surface_elo[p2_surface_key],
            p1_score,
            k=K_FACTOR
        )

        surface_elo[p1_surface_key] = new_p1_surface_elo
        surface_elo[p2_surface_key] = new_p2_surface_elo

        # update forme
        last10_results[p1].append(p1_score)
        last10_results[p2].append(p2_score)

        surface_last10_results[p1_surface_key].append(p1_score)
        surface_last10_results[p2_surface_key].append(p2_score)
        
        pair_key = make_pair_key(p1, p2)
        surface_pair_key = make_surface_pair_key(p1, p2, surface)

        if target == 1:
            winner = str(p1)
        else:
            winner = str(p2)

        # Skupni H2H update
        h2h_matches[pair_key] += 1
        h2h_wins[pair_key][winner] += 1

        # Surface H2H update
        surface_h2h_matches[surface_pair_key] += 1
        surface_h2h_wins[surface_pair_key][winner] += 1

        # Zadnji H2H zmagovalec
        last_h2h_winner[pair_key] = winner

        # update skupnih statistik
        total_matches[p1] += 1
        total_matches[p2] += 1

        total_wins[p1] += p1_score
        total_wins[p2] += p2_score

        # update statistik po površini
        surface_matches[p1_surface_key] += 1
        surface_matches[p2_surface_key] += 1

        surface_wins[p1_surface_key] += p1_score
        surface_wins[p2_surface_key] += p2_score

### Združevanje z osnovnimi značilkami

In [5]:
historical_df = pd.DataFrame(historical_rows).set_index("row_id")

historical_df.tail()

,p1_elo,p2_elo,elo_diff,p1_surface_elo,p2_surface_elo,surface_elo_diff,p1_last10_winrate,p2_last10_winrate,last10_winrate_diff,p1_surface_last10_winrate,...,h2h_p1_wins,h2h_p2_wins,h2h_winrate_p1,h2h_winrate_diff,surface_h2h_matches,surface_h2h_p1_wins,surface_h2h_p2_wins,surface_h2h_winrate_p1,surface_h2h_winrate_diff,last_h2h_result
row_id,,,,,,,,,,,,,,,,,,,,,
68774,1817.572300,1920.864060,-103.291761,1631.257232,1784.189040,-152.931808,0.7,0.9,-0.2,0.600,...,2,1,0.666667,0.333333,0,0,0,0.500000,0.000000,1.0
68775,1673.247257,2285.436223,-612.188966,1616.663262,1974.260352,-357.597089,0.8,1.0,-0.2,0.875,...,0,0,0.500000,0.000000,0,0,0,0.500000,0.000000,0.5
68776,1564.627922,1948.597229,-383.969307,1543.866158,1860.122507,-316.256349,0.6,0.7,-0.1,0.600,...,0,0,0.500000,0.000000,0,0,0,0.500000,0.000000,0.5
68777,1920.864060,2285.436223,-364.572163,1784.189040,1974.260352,-190.071312,0.9,1.0,-0.1,0.900,...,0,1,0.000000,-1.000000,0,0,0,0.500000,0.000000,0.0
68778,1948.597229,2285.436223,-336.838995,1860.122507,1974.260352,-114.137844,0.7,1.0,-0.3,0.700,...,4,9,0.307692,-0.384615,3,1,2,0.333333,-0.333333,0.0


In [6]:
features_final = features.join(historical_df)

features_final.tail()

,tourney_id,tourney_name,tourney_date,year,match_num,p1_id,p1_name,p2_id,p2_name,target,...,h2h_p1_wins,h2h_p2_wins,h2h_winrate_p1,h2h_winrate_diff,surface_h2h_matches,surface_h2h_p1_wins,surface_h2h_p2_wins,surface_h2h_winrate_p1,surface_h2h_winrate_diff,last_h2h_result
68774,2026-1536,Madrid Masters,2026-04-22,2026,365,208103,Jiri Lehecka,209950,Arthur Fils,0,...,2,1,0.666667,0.333333,0,0,0,0.500000,0.000000,1.0
68775,2026-1536,Madrid Masters,2026-04-22,2026,366,212588,Rafael Jodar,206173,Jannik Sinner,0,...,0,0,0.500000,0.000000,0,0,0,0.500000,0.000000,0.5
68776,2026-1536,Madrid Masters,2026-04-22,2026,367,210696,Alexander Blockx,100644,Alexander Zverev,0,...,0,0,0.500000,0.000000,0,0,0,0.500000,0.000000,0.5
68777,2026-1536,Madrid Masters,2026-04-22,2026,368,209950,Arthur Fils,206173,Jannik Sinner,0,...,0,1,0.000000,-1.000000,0,0,0,0.500000,0.000000,0.0
68778,2026-1536,Madrid Masters,2026-04-22,2026,369,100644,Alexander Zverev,206173,Jannik Sinner,0,...,4,9,0.307692,-0.384615,3,1,2,0.333333,-0.333333,0.0


### Hitri preizkusi

Preverimo dimenzije, manjkajoče vrednosti in osnovne korelacije s `target`, da potrdimo, da imajo nove značilke smiseln napovedni signal.

In [7]:
print("features_basic:", features.shape)
print("historical features:", historical_df.shape)
print("features_final:", features_final.shape)

features_basic: (68779, 21)
historical features: (68779, 35)
features_final: (68779, 56)


In [8]:
history_cols = historical_df.columns.tolist()

features_final[history_cols].isna().sum().sort_values(ascending=False).head(20)

p1_elo                         0
h2h_p2_wins                    0
matches_played_diff            0
p1_surface_matches_played      0
p2_surface_matches_played      0
surface_matches_played_diff    0
h2h_matches                    0
h2h_p1_wins                    0
h2h_winrate_p1                 0
p1_matches_played              0
h2h_winrate_diff               0
surface_h2h_matches            0
surface_h2h_p1_wins            0
surface_h2h_p2_wins            0
surface_h2h_winrate_p1         0
surface_h2h_winrate_diff       0
p2_matches_played              0
surface_winrate_diff           0
p2_elo                         0
last10_winrate_diff            0
dtype: int64

In [9]:
features_final[[
    "elo_diff",
    "surface_elo_diff",
    "last10_winrate_diff",
    "surface_winrate_diff",
    "target"
]].corr()["target"].sort_values(ascending=False)

target                  1.000000
elo_diff                0.390502
surface_elo_diff        0.375355
surface_winrate_diff    0.306242
last10_winrate_diff     0.301685
Name: target, dtype: float64

In [10]:
features_final["p1_higher_elo"] = (
    features_final["p1_elo"] > features_final["p2_elo"]
).astype(int)

features_final.groupby("p1_higher_elo")["target"].mean()

p1_higher_elo
0    0.337517
1    0.658498
Name: target, dtype: float64

In [11]:
clay_examples = features_final[
    features_final["surface"] == "Clay"
].copy()

clay_examples[[
    "tourney_date",
    "p1_name",
    "p2_name",
    "surface",
    "p1_surface_elo",
    "p2_surface_elo",
    "surface_elo_diff",
    "target"
]].sort_values("surface_elo_diff", ascending=False).head(10)

,tourney_date,p1_name,p2_name,surface,p1_surface_elo,p2_surface_elo,surface_elo_diff,target
38690,2014-02-17,Rafael Nadal,Joao Sousa,Clay,2309.083295,1458.708219,850.375076,1
39456,2014-05-26,Rafael Nadal,Dusan Lajovic,Clay,2245.707609,1422.874647,822.832962,1
34391,2012-05-27,Rafael Nadal,Denis Istomin,Clay,2267.112557,1445.311845,821.800712,1
41229,2015-02-23,Rafael Nadal,Facundo Arguello,Clay,2240.823565,1427.873397,812.950168,1
46682,2017-04-24,Rafael Nadal,Rogerio Dutra Silva,Clay,2148.278579,1342.927381,805.351199,1
31628,2011-05-08,Rafael Nadal,Paolo Lorenzi,Clay,2258.397371,1454.123887,804.273484,1
36909,2013-05-27,Rafael Nadal,Martin Klizan,Clay,2290.185400,1503.546441,786.638959,1
26570,2009-05-25,Rafael Nadal,Teymuraz Gabashvili,Clay,2239.347357,1453.867228,785.480129,1
31472,2011-04-18,Rafael Nadal,Ivan Dodig,Clay,2270.859326,1491.606364,779.252962,1
41175,2015-02-16,Rafael Nadal,Pablo Carreno Busta,Clay,2267.803615,1494.084761,773.718854,1


### Shranjevanje

In [12]:
h2h_cols = [
    "h2h_matches",
    "h2h_winrate_diff",
    "surface_h2h_matches",
    "surface_h2h_winrate_diff",
    "last_h2h_result"
]

features_final[h2h_cols].describe()

,h2h_matches,h2h_winrate_diff,surface_h2h_matches,surface_h2h_winrate_diff,last_h2h_result
count,68779.000000,68779.000000,68779.000000,68779.000000,68779.000000
mean,1.238096,-0.001677,0.629465,-0.000221,0.499368
std,2.486723,0.566580,1.495736,0.489521,0.338747
min,0.000000,-1.000000,0.000000,-1.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.500000
50%,0.000000,0.000000,0.000000,0.000000,0.500000
75%,2.000000,0.000000,1.000000,0.000000,0.500000
max,55.000000,1.000000,36.000000,1.000000,1.000000


In [13]:
(features_final["h2h_matches"] > 0).mean()

np.float64(0.4589918434405851)

In [14]:
(features_final["surface_h2h_matches"] > 0).mean()

np.float64(0.30708501141336747)

In [15]:
features_final[
    [
        "h2h_matches",
        "h2h_winrate_diff",
        "surface_h2h_matches",
        "surface_h2h_winrate_diff",
        "last_h2h_result",
        "target"
    ]
].corr()["target"].sort_values(ascending=False)

target                      1.000000
h2h_winrate_diff            0.142930
last_h2h_result             0.129766
surface_h2h_winrate_diff    0.118146
surface_h2h_matches         0.001611
h2h_matches                 0.001288
Name: target, dtype: float64

In [16]:
output_path = Path("./data/processed/koncne_znacilke.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

features_final.to_csv(output_path, index=False)

print(features_final.shape)

(68779, 57)
